# Qwen3.5 GDN 状态接口可行性探查

目的：确认 A100 环境、transformers 对 Qwen3.5 的支持，以及 **Gated DeltaNet (GDN) 循环状态**的可访问接口——
这是 TurboQuant vs LeapQuant 状态量化对比实验能否做的前提。用 Qwen3.5-0.8B（同架构、加载快）验证机制。

执行顺序按 cell 顺序；每步打印结论后再决定后续是否加载 9B 跑正式实验。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!pip -q install -U transformers accelerate bitsandbytes 2>&1 | tail -2
import transformers; print('transformers', transformers.__version__)


In [ ]:
# 1) transformers 是否认识 Qwen3.5
import transformers.models as M
mods = [x for x in dir(M) if 'qwen' in x.lower()]
print('qwen modules in transformers:', mods)
from transformers import AutoConfig
for size in ['Qwen/Qwen3.5-0.8B', 'Qwen/Qwen3.5-9B']:
    try:
        cfg = AutoConfig.from_pretrained(size, trust_remote_code=True)
        print('\n==', size, '==', 'model_type=', getattr(cfg, 'model_type', None),
              '| arch=', getattr(cfg, 'architectures', None),
              '| layers=', getattr(cfg, 'num_hidden_layers', None))
        d = cfg.to_dict()
        keys = [k for k in d if any(s in k.lower() for s in ('delta','gate','linear','attn','state','rn','conv'))]
        print('  relevant cfg keys:', keys)
        for k in keys:
            print('   ', k, '=', str(d[k])[:160])
    except Exception as e:
        print('\n', size, 'config error:', type(e).__name__, e)


In [ ]:
# 2) 加载 0.8B，探查 GDN 循环状态接口
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
name = 'Qwen/Qwen3.5-0.8B'
tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(name, trust_remote_code=True,
                                             torch_dtype=torch.bfloat16).to('cuda')
model.eval()
print('model class:', type(model).__module__, type(model).__name__)
# 打印含 delta/gate/state 的子模块
for n, m in model.named_modules():
    ln = n.lower()
    if any(s in ln for s in ('delta','gated','state','linear','conv','mamba','rnn')):
        print('MOD', n, '->', type(m).__name__)


In [ ]:
# 3) 一次前向 + 检查返回结构里是否携带状态（recurrent state / hidden_states / kv）
import torch
inp = tok('The capital of France is', return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model(**inp, use_cache=True)
print('outputs keys:', [k for k in dir(out) if not k.startswith('_')])
print('has past_key_values:', hasattr(out, 'past_key_values'), type(out.past_key_values) if hasattr(out,'past_key_values') else None)
if hasattr(out, 'past_key_values') and out.past_key_values is not None:
    p = out.past_key_values
    print('past_key_values type:', type(p))
    try:
        # 逐层看元素形状
        for i, layer in enumerate(p):
            if isinstance(layer, (tuple, list)):
                print('layer', i, 'len', len(layer), 'shapes', [tuple(t.shape) for t in layer if hasattr(t,'shape')])
            else:
                print('layer', i, '=', type(layer))
    except Exception as e:
        print('iter pv err', e)
else:
    print('WARN: no past_key_values; Qwen3.5 GDN may store state inside model (need hook) or not support cache yet')


In [ ]:
# 4) 若存在 state 张量：定位其 dtype/shape，并确认与 delta rule S += w*v*k^T 一致
import torch
if hasattr(model, 'config'):
    print('model_type =', model.config.model_type)
print('GPU mem (GB):', round(torch.cuda.mem_get_info()[1]/2**30, 1))
print('\n== 探查结束 ==')
print('结论：请根据上面输出判断 —— (a) transformers 是否原生支持 Qwen3.5 GDN；')
print('      (b) 能否通过 past_key_values 拿到/替换每层状态（可做逐窗口/逐步状态量化）；')
print('      (c) 若不能，需要改用 flash-linear-attention(fla) 或自写 GDN forward。')
